In [1]:
import os, subprocess, shutil

VECTORSTORE_DIR = os.path.expanduser("~/tisd/vectorstore")

# Fix folder + all file permissions
subprocess.run(["chmod", "-R", "755", VECTORSTORE_DIR], check=True)
print("Permissions fixed")

# Remove ALL stale db files — fresh start
for fname in ["chroma.sqlite3", "faiss_index.bin", "chunk_metadata.json"]:
    fpath = os.path.join(VECTORSTORE_DIR, fname)
    if os.path.exists(fpath):
        os.remove(fpath)
        print(f"Removed {fname}")

# Confirm clean state
print(f"\nvectorstore/ now contains:")
for f in os.listdir(VECTORSTORE_DIR):
    print(f"  {f}")
# Should print nothing, or just subdirs

Permissions fixed
Removed faiss_index.bin
Removed chunk_metadata.json

vectorstore/ now contains:


In [2]:
import time, json, os, re, sys
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

VECTORSTORE_DIR = os.path.expanduser("~/tisd/vectorstore")
CHUNKS_PATH     = os.path.expanduser("~/tisd/data/processed/chunks.json")
COLLECTION_NAME = "tisd_knowledge"
BATCH_SIZE      = 64

# ── ChromaDB fresh ───────────────────────────────────
client = chromadb.PersistentClient(path=VECTORSTORE_DIR)

existing = [c.name for c in client.list_collections()]
print(f"Existing collections: {existing}")

if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted old '{COLLECTION_NAME}'")

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)
print(f"Created '{COLLECTION_NAME}'")

# ── Chunks ───────────────────────────────────────────
with open(CHUNKS_PATH) as f:
    chunks = json.load(f)
print(f"Loaded {len(chunks)} chunks")

# ── Embedder ─────────────────────────────────────────
try:
    embedder
    print("Reusing embedder")
except NameError:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    print("Loaded embedder")

# ── Build texts / metadatas / ids ────────────────────
texts, metadatas, ids = [], [], []

for i, c in enumerate(chunks):
    if c.get("word_count", 0) < 20:
        continue

    text = c["chunk_text"].strip()
    text = re.sub(r'Chapter \d+\.indd \d+', '', text)
    text = re.sub(r'\d{2}-\d{2}-\d{4} \d{2}:\d{2}:\d{2}', '', text)
    text = re.sub(r'Reprint \d{4}-\d{2,4}', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if len(text) < 50:
        continue

    try:
        class_int = int(c.get("class_level", 4))
    except (ValueError, TypeError):
        class_int = 4

    meta = {
        "filename":    str(c.get("filename",    "unknown")),
        "source_type": str(c.get("source_type", "textbook")),
        "class_level": class_int,          # int — required for $lte
        "subject":     str(c.get("subject", "general")),
        "page_num":    int(c.get("page_num",    0)),
        "word_count":  int(c.get("word_count",  0)),
    }

    texts.append(text)
    metadatas.append(meta)
    ids.append(f"chunk_{i}")

print(f"After cleaning: {len(texts)} chunks "
      f"(dropped {len(chunks) - len(texts)})")

# ── Embed ─────────────────────────────────────────────
t0 = time.time()
all_embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding"):
    batch = texts[i : i + BATCH_SIZE]
    embs  = embedder.encode(
        batch,
        show_progress_bar=False,
        convert_to_numpy=True
    ).tolist()
    all_embeddings.extend(embs)

print(f"Embedding: {time.time()-t0:.1f}s")

# ── Index ─────────────────────────────────────────────
t0 = time.time()

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Indexing"):
    collection.add(
        ids        = ids           [i : i + BATCH_SIZE],
        embeddings = all_embeddings[i : i + BATCH_SIZE],
        documents  = texts         [i : i + BATCH_SIZE],
        metadatas  = metadatas     [i : i + BATCH_SIZE],
    )

print(f"Indexing: {time.time()-t0:.1f}s")
print(f"\nFinal count: {collection.count()} documents")
assert collection.count() > 0
print("Vectorstore ready.")

Existing collections: []
Created 'tisd_knowledge'
Loaded 6422 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedder
After cleaning: 6199 chunks (dropped 223)


Embedding:   0%|          | 0/97 [00:00<?, ?it/s]

Embedding: 34.3s


Indexing:   0%|          | 0/97 [00:00<?, ?it/s]

Indexing: 3.8s

Final count: 6199 documents
Vectorstore ready.


In [3]:
q_emb = embedder.encode("How do people travel?").tolist()

results = collection.query(
    query_embeddings=[q_emb],
    n_results=3,
    where={"class_level": {"$lte": 4}},   # int — no quotes
    include=["documents", "metadatas", "distances"]
)

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(f"[class {meta['class_level']} | {meta['subject']} | dist {dist:.3f}]")
    print(f"{doc[:150]}\n")

[class 4 | general | dist 0.516]
Page 416 instruction. A child's whole future life, to a large extent, depends on the teaching it receives in early childhood, and it is needless to sa

[class 2 | English | dist 0.567]
Take a bus Or take a train, Take a boat Or take a plane, Take a taxi, Take a car, Maybe near Or maybe far, Sight words take | is | the | or | two | ma

[class 4 | Evs | dist 0.576]
22 Our Wondrous World “Some flyovers are built for metros, and others are for buses and cars”, explained Dada ji. Dada ji continued, “These modern fac



In [4]:
for key in list(sys.modules.keys()):
    if "tisd_engine_mlx" in key:
        del sys.modules[key]

from tisd_engine_mlx import TISDEngine
engine = TISDEngine(verbose=True)
engine.load()

ModuleNotFoundError: No module named 'tisd_engine_mlx'

In [14]:
%%writefile tisd_engine_mlx.py
# ~/tisd/notebooks/tisd_engine_mlx.py
import os
import time
import json
import psutil
import mlx.core as mx
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer

load_dotenv()

# --- CONFIG ---
MODEL_ID       = "mlx-community/Phi-3-mini-4k-instruct-4bit"
EMBED_MODEL_ID = "all-MiniLM-L6-v2"

# FIX: Path relative to this file to find the vectorstore in the parent directory
BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
VECTORSTORE_DIR = os.path.join(BASE_DIR, "vectorstore")

COLLECTION_NAME = "tisd_knowledge"
MAX_TOKENS      = 300
TOP_K_RETRIEVE  = 15
TOP_K_RERANK    = 3
MAX_GRADE       = 4

SAMPLER_CONFIG = {
    "temp": 0.0,
    "top_p": 1.0,
    "min_p": 0.0,
    "min_tokens_to_keep": 1,
}

SYSTEM_PROMPT = """You are Tara, a warm and patient teacher for children in Grade 1 to 4.
Rules you must always follow:
1. Use simple words a young child understands.
2. Keep your answer to 3-4 short sentences maximum.
3. Only use information from the CONTEXT provided below.
4. If the context does not contain the answer, say: "That's a great question! Let's ask your teacher about that one."
5. Never make up facts. Never use complicated words.
6. End with one encouraging sentence."""

def get_memory_stats() -> dict:
    vm   = psutil.virtual_memory()
    swap = psutil.swap_memory()
    return {
        "ram_used_gb":  round(vm.used   / 1e9, 2),
        "ram_avail_gb": round(vm.available / 1e9, 2),
        "swap_used_gb": round(swap.used  / 1e9, 2),
        "ram_pct":      vm.percent,
    }

class TISDEngine:
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
        self.model      = None
        self.tokenizer  = None
        self.embedder   = None
        self.collection = None
        self.sampler    = None

    def load(self):
        t0 = time.time()
        if self.verbose: print("Loading embedding model...")
        self.embedder = SentenceTransformer(EMBED_MODEL_ID)

        if self.verbose: print(f"Connecting to ChromaDB at {VECTORSTORE_DIR}...")
        client = chromadb.PersistentClient(path=VECTORSTORE_DIR)
        self.collection = client.get_collection(COLLECTION_NAME)

        if self.verbose: print(f"Loading {MODEL_ID} with MLX...")
        self.model, self.tokenizer = load(MODEL_ID)
        self.sampler = make_sampler(**SAMPLER_CONFIG)

        elapsed = time.time() - t0
        mem = get_memory_stats()
        if self.verbose:
            print(f"\nEngine ready in {elapsed:.1f}s")
            print(f"RAM: {mem['ram_used_gb']}GB used | Swap: {mem['swap_used_gb']}GB")

    def _retrieve(self, question: str, grade: int) -> list[str]:
        q_embedding = self.embedder.encode(question).tolist()
        results = self.collection.query(
            query_embeddings=[q_embedding],
            n_results=TOP_K_RETRIEVE,
            where={"class_level": {"$lte": int(grade)}},
            include=["documents", "metadatas", "distances"]
        )
        docs = results["documents"][0]
        distances = results["distances"][0]
        ranked = sorted(zip(distances, docs), key=lambda x: x[0])
        return [doc for _, doc in ranked[:TOP_K_RERANK]]

    def _build_prompt(self, question: str, chunks: list[str]) -> str:
        context = "\n\n---\n\n".join(chunks)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nSTUDENT QUESTION: {question}"},
        ]
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def ask(self, question: str, grade: int = 4, verbose: bool = None) -> dict:
        if verbose is None: verbose = self.verbose
        t_start = time.time()
        
        chunks = self._retrieve(question, grade)
        if not chunks:
            return {"answer": "I don't have that in my books yet!", "latency_ms": {"total": 0}}

        prompt = self._build_prompt(question, chunks)
        
        t0 = time.time()
        answer = generate(self.model, self.tokenizer, prompt=prompt, max_tokens=MAX_TOKENS, sampler=self.sampler, verbose=False)
        t_gen = time.time() - t0

        answer = answer.replace("<|end|>", "").replace("<|endoftext|>", "").strip()
        t_total = time.time() - t_start

        if verbose:
            print(f"\nAnswer: {answer}")
            print(f"Latency: {round(t_total*1000)}ms")

        return {"answer": answer, "latency_ms": {"total": round(t_total*1000)}, "memory": get_memory_stats()}

Overwriting tisd_engine_mlx.py


In [15]:
# Cell: Load and Test
import sys
import os

# Ensure the notebooks directory is in the path
sys.path.append(os.getcwd())

# Force reload logic
for key in list(sys.modules.keys()):
    if "tisd_engine_mlx" in key:
        del sys.modules[key]

from tisd_engine_mlx import TISDEngine

# Initialize the Native M4 Engine
engine = TISDEngine(verbose=True)
engine.load()

# TEST QUERY
print("\n--- TESTING THE REVOLUTION ---")
query = "What is the sun and why is it hot?"
result = engine.ask(query, grade=4)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting to ChromaDB at /Users/bhushan/tisd/vectorstore...
Loading mlx-community/Phi-3-mini-4k-instruct-4bit with MLX...


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]


Engine ready in 5.8s
RAM: 5.86GB used | Swap: 1.17GB

--- TESTING THE REVOLUTION ---


In [16]:
# Updated Cell: Test with Streaming to see the M4 in action!
from mlx_lm import stream

def ask_streaming(engine, question, grade=4):
    print(f"🧑‍🎓 Query: {question}")
    print("🤖 Tara: ", end="", flush=True)
    
    # 1. Retrieve
    chunks = engine._retrieve(question, grade)
    prompt = engine._build_prompt(question, chunks)
    
    # 2. Stream the generation directly to the screen
    # This bypasses the 'wait for full answer' delay
    t0 = time.time()
    for response in stream(engine.model, engine.tokenizer, prompt=prompt, max_tokens=300, temp=0.0):
        print(response.text, end="", flush=True)
    
    print(f"\n\n⏱️ Total Time: {time.time() - t0:.2f}s")

# NOW TRY THE STREAMING VERSION
ask_streaming(engine, "What is the sun and why is it hot?")

ImportError: cannot import name 'stream' from 'mlx_lm' (/opt/homebrew/Caskroom/miniforge/base/envs/tisd/lib/python3.11/site-packages/mlx_lm/__init__.py)

In [17]:
%%writefile tisd_engine_mlx.py
import os
import time
import json
import psutil
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer

load_dotenv()

MODEL_ID       = "mlx-community/Phi-3-mini-4k-instruct-4bit"
EMBED_MODEL_ID = "all-MiniLM-L6-v2"
BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
VECTORSTORE_DIR = os.path.join(BASE_DIR, "vectorstore")
COLLECTION_NAME = "tisd_knowledge"

# DETERMINISTIC SAMPLER
SAMPLER_CONFIG = {"temp": 0.0}

SYSTEM_PROMPT = """You are Tara, a warm and patient teacher. 
Use ONLY the CONTEXT to answer in 2-3 short sentences for a child.
If you don't know, say: 'Let's ask your teacher!'"""

def get_memory_stats():
    vm = psutil.virtual_memory()
    return {"ram_used": round(vm.used / 1e9, 2), "swap": round(psutil.swap_memory().used / 1e9, 2)}

class TISDEngine:
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.model, self.tokenizer, self.embedder, self.collection = None, None, None, None
        self.sampler = make_sampler(**SAMPLER_CONFIG)

    def load(self):
        self.embedder = SentenceTransformer(EMBED_MODEL_ID)
        client = chromadb.PersistentClient(path=VECTORSTORE_DIR)
        self.collection = client.get_collection(COLLECTION_NAME)
        self.model, self.tokenizer = load(MODEL_ID)
        if self.verbose: print(f"✅ Engine Ready. RAM: {get_memory_stats()['ram_used']}GB")

    def ask(self, question, grade=4):
        # 1. Retrieve
        q_emb = self.embedder.encode(question).tolist()
        res = self.collection.query(query_embeddings=[q_emb], n_results=3, where={"class_level": {"$lte": int(grade)}})
        context = "\n\n".join(res["documents"][0])
        
        # 2. Build Prompt
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        # 3. Generate (With verbose=True to force streaming to the terminal/output)
        print(f"\n🧑‍🎓 Student: {question}\n🤖 Tara: ", end="", flush=True)
        
        answer = generate(
            self.model, 
            self.tokenizer, 
            prompt=prompt, 
            max_tokens=150, 
            sampler=self.sampler, 
            verbose=True # <-- THIS ENABLES NATIVE MLX STREAMING
        )
        return answer

Overwriting tisd_engine_mlx.py


In [18]:
import sys, os
sys.path.append(os.getcwd())

# Force clear cache
for key in list(sys.modules.keys()):
    if "tisd_engine_mlx" in key: del sys.modules[key]

from tisd_engine_mlx import TISDEngine
engine = TISDEngine(verbose=True)
engine.load()

# THE BIG MOMENT
print("\n--- INFERENCE START ---")
engine.ask("What is the sun and why is it hot?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Engine Ready. RAM: 6.48GB

--- INFERENCE START ---

🧑‍🎓 Student: What is the sun and why is it hot?
🤖 Tara: ==========
The Sun is a star, a massive ball of burning gas that is constantly undergoing nuclear reactions, releasing a tremendous amount of energy. This energy is what makes the Sun hot. It's also the reason why the Sun is the center of our solar system, providing light and heat to Earth and the other planets.<|end|>
Prompt: 532 tokens, 223.994 tokens-per-sec
Generation: 72 tokens, 41.416 tokens-per-sec
Peak memory: 2.996 GB


"The Sun is a star, a massive ball of burning gas that is constantly undergoing nuclear reactions, releasing a tremendous amount of energy. This energy is what makes the Sun hot. It's also the reason why the Sun is the center of our solar system, providing light and heat to Earth and the other planets.<|end|>"

In [3]:
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sentence_transformers import util

# 1. Load the Test Set
TEST_DATA_PATH = "../data/test_set.json"
with open(TEST_DATA_PATH, "r") as f:
    test_questions = json.load(f)

print(f"📊 Starting Evaluation on {len(test_questions)} questions...")

results = []

# 2. Evaluation Loop
for item in tqdm(test_questions, desc="Testing Tara"):
    q = item["question"]
    gt = item["ground_truth"]
    
    # Run the Native M4 Engine (Turning off internal prints for clean table)
    # We create a temporary wrapper to suppress the 'verbose=True' output if needed,
    # but for now, let's just capture the result.
    out = engine.ask(q, verbose=False)
    answer = out # Since the current .ask returns the string answer
    
    # 3. Calculate Semantic Score (Meaning vs Meaning)
    # We use the embedder already loaded in the engine!
    emb_pred = engine.embedder.encode(answer, convert_to_tensor=True)
    emb_gt = engine.embedder.encode(gt, convert_to_tensor=True)
    
    score = util.cos_sim(emb_pred, emb_gt).item()
    
    results.append({
        "Question": q,
        "Ground Truth": gt,
        "Tara's Answer": answer,
        "Semantic Score": round(score, 4)
    })

# 4. Display Results
df_eval = pd.DataFrame(results)
avg_score = df_eval["Semantic Score"].mean()

# Formatting for the report
from IPython.display import display, HTML
display(HTML(f"""
<div style="background-color: #1a1a1a; color: #00ff00; padding: 20px; border-radius: 10px; border: 2px solid #00ff00; font-family: monospace;">
    <h2 style="color: #00ff00; margin-top: 0;">🚀 TISD NATIVE EVALUATION REPORT</h2>
    <p style="font-size: 20px;"><b>Avg Semantic Accuracy: {avg_score:.2%}</b></p>
    <p>Hardware: Apple M4 (Unified Memory)</p>
    <p>Model: Phi-3-mini-4bit (MLX)</p>
</div>
"""))

display(df_eval.sort_values(by="Semantic Score", ascending=False))

📊 Starting Evaluation on 3 questions...


Testing Tara:   0%|          | 0/3 [00:00<?, ?it/s]

TypeError: TISDEngine.ask() got an unexpected keyword argument 'verbose'

In [21]:
%%writefile tisd_engine_mlx 
import os
import time
import json
import psutil
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer

load_dotenv()

# --- CONFIG ---
MODEL_ID       = "mlx-community/Phi-3-mini-4k-instruct-4bit"
EMBED_MODEL_ID = "all-MiniLM-L6-v2"
BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
VECTORSTORE_DIR = os.path.join(BASE_DIR, "vectorstore")
COLLECTION_NAME = "tisd_knowledge"

SAMPLER_CONFIG = {"temp": 0.0}

SYSTEM_PROMPT = """You are Tara, a warm and patient teacher. 
Use ONLY the CONTEXT to answer in 2-3 short sentences for a child.
If you don't know, say: 'Let's ask your teacher!'"""

class TISDEngine:
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.model, self.tokenizer, self.embedder, self.collection = None, None, None, None
        self.sampler = make_sampler(**SAMPLER_CONFIG)

    def load(self):
        self.embedder = SentenceTransformer(EMBED_MODEL_ID)
        client = chromadb.PersistentClient(path=VECTORSTORE_DIR)
        self.collection = client.get_collection(COLLECTION_NAME)
        self.model, self.tokenizer = load(MODEL_ID)
        print(f"✅ Engine Ready. RAM: {round(psutil.virtual_memory().used / 1e9, 2)}GB")

    def ask(self, question, grade=4, verbose=None):
        # Use class default if not specified
        if verbose is None:
            verbose = self.verbose
            
        # 1. Retrieve
        q_emb = self.embedder.encode(question).tolist()
        res = self.collection.query(query_embeddings=[q_emb], n_results=3, where={"class_level": {"$lte": int(grade)}})
        context = "\n\n".join(res["documents"][0])
        
        # 2. Build Prompt
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        # 3. Generate
        if verbose:
            print(f"\n🧑‍🎓 Student: {question}\n🤖 Tara: ", end="", flush=True)
        
        # MLX generate uses 'verbose' to stream to the console
        answer = generate(
            self.model, 
            self.tokenizer, 
            prompt=prompt, 
            max_tokens=150, 
            sampler=self.sampler, 
            verbose=verbose 
        )
        return answer.strip()

Writing tisd_engine_mlx


In [2]:
# Cell: Initialize Engine
import sys
import os
import time

# Ensure current directory is in path
sys.path.append(os.getcwd())

# Force reload the new tisd_engine_mlx.py file
import importlib
import tisd_engine_mlx
importlib.reload(tisd_engine_mlx)
from tisd_engine_mlx import TISDEngine

# Initialize
engine = TISDEngine(verbose=True)
engine.load()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Engine Ready. RAM: 6.19GB


In [8]:
# Cell: Semantic Evaluation Suite
import json
import pandas as pd
from tqdm.notebook import tqdm
from sentence_transformers import util
from IPython.display import display, HTML

# 1. Load the Test Set
TEST_DATA_PATH = "../data/test_set.json"
with open(TEST_DATA_PATH, "r") as f:
    test_questions = json.load(f)

print(f"📊 Starting Evaluation on {len(test_questions)} questions...")

results = []

# 2. Evaluation Loop
for item in tqdm(test_questions, desc="Testing Tara"):
    q = item["question"]
    gt = item["ground_truth"]
    
    # We pass verbose=False to the ask function for a clean table
    answer = engine.ask(q, verbose=False)
    
    # 3. Calculate Semantic Score
    emb_pred = engine.embedder.encode(answer, convert_to_tensor=True)
    emb_gt = engine.embedder.encode(gt, convert_to_tensor=True)
    score = util.cos_sim(emb_pred, emb_gt).item()
    
    results.append({
        "Question": q,
        "Ground Truth": gt,
        "Tara's Answer": answer,
        "Semantic Score": round(score, 4)
    })

# 4. Display Report
df_eval = pd.DataFrame(results)
avg_score = df_eval["Semantic Score"].mean()

display(HTML(f"""
<div style="background-color: #1a1a1a; color: #00ff00; padding: 20px; border-radius: 10px; border: 2px solid #00ff00;">
    <h2 style="color: #00ff00; margin-top: 0;">🚀 TISD NATIVE EVALUATION REPORT</h2>
    <p style="font-size: 20px;"><b>Avg Semantic Accuracy: {avg_score:.2%}</b></p>
</div>
"""))

display(df_eval.sort_values(by="Semantic Score", ascending=False))

📊 Starting Evaluation on 3 questions...


Testing Tara:   0%|          | 0/3 [00:00<?, ?it/s]

,Question,Ground Truth,Tara's Answer,Semantic Score
2,What are sense organs?,"Sense organs are body parts like eyes, ears, n...",Sense organs are special parts of your body th...,0.9362
1,What do plants need to grow?,"Plants need sunlight, water, and air to grow.","Plants need sunlight, water, carbon dioxide, a...",0.8727
0,What is the sun?,The sun is the brightest object in the sky and...,"The Sun is a star, a massive ball of burning g...",0.5780


In [5]:
# Cell: FORCE REFRESH ENGINE
import sys
import os
import importlib

# 1. Manually wipe the old engine from Python's memory cache
if "tisd_engine_mlx" in sys.modules:
    del sys.modules["tisd_engine_mlx"]

# 2. Re-import the fresh file from the disk
import tisd_engine_mlx
from tisd_engine_mlx import TISDEngine

# 3. Re-initialize the brain
engine = TISDEngine(verbose=True)
engine.load()

print("✅ NEW Engine loaded into memory. Testing 'verbose' parameter...")
# Quick test to prove it works now:
try:
    engine.ask("Test", verbose=False)
    print("🚀 SUCCESS: 'verbose' parameter is now recognized!")
except TypeError as e:
    print(f"❌ Still failing: {e}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Engine Ready. RAM: 6.6GB
✅ NEW Engine loaded into memory. Testing 'verbose' parameter...
❌ Still failing: TISDEngine.ask() got an unexpected keyword argument 'verbose'


In [6]:
%%writefile tisd_engine_mlx.py
import os
import time
import json
import psutil
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer

load_dotenv()

# --- CONFIG ---
MODEL_ID       = "mlx-community/Phi-3-mini-4k-instruct-4bit"
EMBED_MODEL_ID = "all-MiniLM-L6-v2"
BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__name__), ".."))
VECTORSTORE_DIR = os.path.join(BASE_DIR, "vectorstore")
COLLECTION_NAME = "tisd_knowledge"

SAMPLER_CONFIG = {"temp": 0.0}

SYSTEM_PROMPT = """You are Tara, a warm and patient teacher. 
Use ONLY the CONTEXT to answer in 2-3 short sentences for a child.
If you don't know, say: 'Let's ask your teacher!'"""

class TISDEngine:
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.model, self.tokenizer, self.embedder, self.collection = None, None, None, None
        self.sampler = make_sampler(**SAMPLER_CONFIG)

    def load(self):
        self.embedder = SentenceTransformer(EMBED_MODEL_ID)
        client = chromadb.PersistentClient(path=VECTORSTORE_DIR)
        self.collection = client.get_collection(COLLECTION_NAME)
        self.model, self.tokenizer = load(MODEL_ID)
        print(f"✅ Engine Ready. RAM: {round(psutil.virtual_memory().used / 1e9, 2)}GB")

    def ask(self, question, grade=4, verbose=None):
        # FIX: The parameter 'verbose' is now explicitly in the signature!
        if verbose is None:
            verbose = self.verbose
            
        # 1. Retrieve
        q_emb = self.embedder.encode(question).tolist()
        res = self.collection.query(query_embeddings=[q_emb], n_results=3, where={"class_level": {"$lte": int(grade)}})
        context = "\n\n".join(res["documents"][0])
        
        # 2. Build Prompt
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        # 3. Generate
        if verbose:
            print(f"\n🧑‍🎓 Student: {question}\n🤖 Tara: ", end="", flush=True)
        
        answer = generate(
            self.model, 
            self.tokenizer, 
            prompt=prompt, 
            max_tokens=150, 
            sampler=self.sampler, 
            verbose=verbose 
        )
        return answer.strip()

Overwriting tisd_engine_mlx.py


In [7]:
# Cell: Load and Verify
import importlib
import tisd_engine_mlx
importlib.reload(tisd_engine_mlx) # The "Surgical Strike"
from tisd_engine_mlx import TISDEngine

# Initialize
engine = TISDEngine(verbose=True)
engine.load()

# THE TEST: If this prints "IT WORKS", the nightmare is over.
print("\n🧪 Testing 'verbose' parameter...")
try:
    engine.ask("Can you hear me?", verbose=False)
    print("🚀 SUCCESS: The engine is stabilized!")
except TypeError as e:
    print(f"❌ Still seeing error: {e}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Engine Ready. RAM: 7.18GB

🧪 Testing 'verbose' parameter...
🚀 SUCCESS: The engine is stabilized!
